# 🧠 Aula 07 — Introdução ao Modelo CUDA

**Objetivo:** compreender a estrutura de programação paralela do CUDA — kernels, threads,
blocos e grades — e aplicá-la para acelerar cálculos intensivos em IA, medindo o speedup
real contra a CPU.

**Roteiro deste notebook:**
1. Verificação do ambiente (há GPU CUDA?).
2. Teoria: host/device, kernels e o fluxo CUDA.
3. A hierarquia: índice global (grades 1D e 2D).
4. Primeiro kernel: soma de vetores.
5. Atividade: FFT CPU (NumPy) vs. GPU (CuPy).
6. Discussão e síntese.

> ⚠️ **Requer GPU NVIDIA (CUDA).** Sem GPU, as células mostram o conceito e os números de
> referência — a aula roda do começo ao fim. Ative a T4: *Runtime ➔ Change runtime type*.

## 1. Verificação do Ambiente

CUDA só roda em **GPU NVIDIA**. Vamos confirmar se o `numba.cuda` encontra uma.

In [ ]:
# @title 🔍 Há GPU CUDA disponível?
# ============================================================================
# OBJETIVO: checar se o numba.cuda encontra uma GPU NVIDIA.
# Isso define se os kernels abaixo rodam de verdade ou mostram a referência.
# ============================================================================
try:
    from numba import cuda
    TEM_CUDA = cuda.is_available()
    if TEM_CUDA:
        print(f"✅ CUDA disponível: {cuda.get_current_device().name.decode()}")
    else:
        print("⚠️  Sem GPU CUDA — os kernels mostrarão o conceito e a referência.")
except ImportError:
    TEM_CUDA = False
    print("⚠️  Numba não instalado. No Colab:  !pip install numba -q")

print("Para GPU real: Runtime ➔ Change runtime type ➔ T4 GPU.")

## 2. Teoria: o modelo de programação CUDA

**CUDA** (Compute Unified Device Architecture) é a plataforma da NVIDIA para computação
paralela. O programador escreve **kernels** — funções executadas em paralelo por milhares
de threads na GPU.

| | **Host (CPU)** | **Device (GPU)** |
| :--- | :--- | :--- |
| Papel | Aloca, transfere, lança kernels, coleta | Executa o kernel em N threads |
| Memória | RAM | VRAM |

**Fluxo típico:**
1. `cuda.to_device()` — CPU aloca arrays na VRAM;
2. `kernel[blocos, threads](...)` — CPU lança o kernel;
3. GPU executa N threads em paralelo;
4. `cuda.synchronize()` — CPU espera a GPU terminar;
5. `copy_to_host()` — CPU copia o resultado de volta.

## 3. A hierarquia: o índice global

| Identificador | O que é |
| :--- | :--- |
| `threadIdx` | Índice da thread **dentro** do bloco |
| `blockIdx` | Índice do bloco **dentro** da grade |
| `blockDim` | Nº de threads por bloco |
| `gridDim` | Nº de blocos na grade |

**Fórmula do índice global (1D):** `idx = blockIdx.x * blockDim.x + threadIdx.x`, ou o
atalho `cuda.grid(1)`. Em 2D: `col, row = cuda.grid(2)`.

In [ ]:
# @title 🔢 Índices globais em grades 1D e 2D
# ============================================================================
# OBJETIVO: ver a fórmula do índice global em ação (cada thread escreve
# seu próprio índice) e um grid 2D para coordenadas (linha/coluna).
# ============================================================================
if TEM_CUDA:
    from numba import cuda
    import numpy as np

    @cuda.jit
    def kernel_1d(arr):
        idx = cuda.grid(1)                 # blockIdx*blockDim + threadIdx
        if idx < arr.shape[0]:
            arr[idx] = idx

    N = 32
    arr = np.zeros(N, dtype=np.int32)
    arr_d = cuda.to_device(arr)
    kernel_1d[1, N](arr_d)                 # 1 bloco, 32 threads
    cuda.synchronize()
    print("Índices 1D:", arr_d.copy_to_host().tolist())

    @cuda.jit
    def kernel_2d(matriz):
        col, row = cuda.grid(2)            # x = coluna, y = linha
        if col < matriz.shape[1] and row < matriz.shape[0]:
            matriz[row, col] = row * 1000 + col

    H, W = 4, 8
    mat = np.zeros((H, W), dtype=np.int32)
    mat_d = cuda.to_device(mat)
    kernel_2d[(1, 1), (W, H)](mat_d)
    cuda.synchronize()
    print("Índices 2D (row*1000 + col):")
    print(mat_d.copy_to_host())
else:
    print("Sem CUDA. A fórmula distribui os índices assim (3 blocos x 4 threads):")
    for bloco in range(3):
        print(f"  bloco {bloco}: ", [bloco * 4 + t for t in range(4)])

## 4. Primeiro kernel: soma de vetores

Cada thread calcula **uma posição** do vetor. É o "olá, mundo" do CUDA.

In [ ]:
# @title ➕ Primeiro kernel: soma de vetores (host -> device -> host)
# ============================================================================
# OBJETIVO: percorrer o fluxo CUDA completo, medindo o resultado.
# ============================================================================
if TEM_CUDA:
    from numba import cuda
    import numpy as np

    @cuda.jit
    def soma_vetores_gpu(a, b, c):
        idx = cuda.grid(1)                 # cada thread = uma posição
        if idx < c.shape[0]:
            c[idx] = a[idx] + b[idx]

    N = 10_000_000
    a = np.ones(N, dtype=np.float32)
    b = np.ones(N, dtype=np.float32) * 2
    c = np.zeros(N, dtype=np.float32)

    a_d = cuda.to_device(a)                # RAM -> VRAM
    b_d = cuda.to_device(b)
    c_d = cuda.to_device(c)

    threads_por_bloco = 256
    blocos = (N + threads_por_bloco - 1) // threads_por_bloco
    print(f"N = {N:,} | {threads_por_bloco} threads/bloco | {blocos:,} blocos")

    soma_vetores_gpu[blocos, threads_por_bloco](a_d, b_d, c_d)
    cuda.synchronize()                     # espera a GPU terminar
    resultado = c_d.copy_to_host()         # VRAM -> RAM
    print(f"Resultado[0..4] = {resultado[:5]}  (esperado [3. 3. 3. 3. 3.])")
else:
    print("Sem CUDA — resultado esperado: [3. 3. 3. 3. 3.]")
    print("Com N=10M e 256 threads/bloco, a grade tem 39.063 blocos.")

## 5. Atividade: FFT — CPU vs. GPU

Agora o problema real: calcular a **Transformada de Fourier** (FFT) de milhões de amostras
de áudio. Comparamos **NumPy (CPU)** com **CuPy (GPU)** — a mesma API, engine diferente.

In [ ]:
# @title 🌊 FFT: CPU (NumPy) vs. GPU (CuPy)
# ============================================================================
# OBJETIVO: medir o speedup da FFT na GPU — a operação central do
# pré-processamento de áudio para reconhecimento de fala.
# ============================================================================
import numpy as np, time

N = 2**22   # ~4 milhões de pontos
sinal_cpu = np.random.randn(N).astype(np.float32)

# ── CPU (NumPy) ─────────────────────────────────────────────────────────────
inicio = time.perf_counter()
fft_cpu = np.fft.fft(sinal_cpu)
t_cpu = time.perf_counter() - inicio
print(f"CPU (NumPy FFT): {t_cpu*1000:8.2f} ms")

# ── GPU (CuPy) ──────────────────────────────────────────────────────────────
try:
    import cupy as cp
    sinal_gpu = cp.asarray(sinal_cpu)          # RAM -> VRAM
    _ = cp.fft.fft(sinal_gpu)                  # warm-up (descartado)
    cp.cuda.Stream.null.synchronize()

    inicio = time.perf_counter()
    fft_gpu = cp.fft.fft(sinal_gpu)
    cp.cuda.Stream.null.synchronize()          # espera a GPU terminar
    t_gpu = time.perf_counter() - inicio
    print(f"GPU (CuPy FFT) : {t_gpu*1000:8.2f} ms")
    print(f"Speedup        : {t_cpu/t_gpu:.1f}x mais rápido!")

    # Validação: as duas FFTs devem coincidir (com tolerância).
    assert cp.allclose(cp.asarray(np.fft.fft(sinal_cpu)), fft_gpu, atol=1e-1)
    print("✅ FFT CPU == FFT GPU (dentro da tolerância).")
except ImportError:
    print("CuPy não disponível — instale: pip install cupy-cuda12x")
    print("Referência (T4): GPU ~18 ms vs. CPU ~1200 ms (~66x).")

In [ ]:
# @title 📐 Escolher o tamanho de bloco (threads/bloco)
# ============================================================================
# OBJETIVO: quantos blocos cada tamanho de bloco gera e quanto 'sobra'.
# ============================================================================
import math

N = 1_000_000
print(f"{'threads/bloco':>13} | {'blocos':>8} | {'threads totais':>14} | overhead")
print("-" * 60)
for tpb in (32, 64, 128, 256, 512, 1024):
    bpg = math.ceil(N / tpb)
    total = bpg * tpb
    extra = "  <- ótimo" if tpb == 256 else ""
    print(f"{tpb:>13} | {bpg:>8} | {total:>14,} | {total - N:>7,}{extra}")
print()
print("Regra: múltiplo de 32 (warp); 128–256 é o ponto ótimo; máx 1024.")

## 6. Discussão em Grupo

Em grupos de 3–4, no cenário da startup de áudio:

1. 10M de amostras com 256 threads/bloco: quantos blocos? E as threads 'extras'?
2. Por que `cuda.synchronize()` é necessário antes de copiar o resultado?
3. Em que situações um kernel CUDA simples **não** teria speedup vs. a CPU?
4. O CuPy dá 66× na FFT. Como você usaria isso no pipeline da startup?

> Atividade de pesquisa completa em `aulas/aula07/atividade.md`.

## 7. Exercícios (5)

Resolva os 5 exercícios **neste notebook** (Colab com GPU T4). O valor está em
**experimentar e explicar**.

---

**1) O índice global.** Explique a fórmula `idx = blockIdx.x * blockDim.x + threadIdx.x`.
Por que ela garante um índice **único** por thread?

**2) Escolha de bloco.** Para N = 1.000.000 com 256 threads/bloco, quantos blocos são
lançados? Por que usamos `(N + threads - 1) // threads` (e não `N // threads`)?

**3) Medindo um kernel.** Na célula-esqueleto, complete um kernel de soma de vetores e
meça o tempo com `cuda.event`. Compare com a versão NumPy na CPU.

**4) Sincronização.** O que acontece se você copiar o resultado com `copy_to_host()` **sem**
`cuda.synchronize()`? Por que isso é um erro?

**5) FFT e o pipeline.** O CuPy dá ~66× de speedup na FFT. Descreva como você encaixaria
isso no pipeline de **áudio** da startup para atingir latência abaixo de 100 ms.


In [ ]:
# @title Exercício 3 — complete o kernel e meça com cuda.event
# ============================================================================
# OBJETIVO: escrever o kernel de soma de vetores e medir o tempo na GPU.
# Requer GPU NVIDIA (Colab). Sem GPU, a celula explica o resultado esperado.
# ============================================================================
try:
    from numba import cuda
    import numpy as np
    if not cuda.is_available():
        raise RuntimeError("sem GPU CUDA")

    @cuda.jit
    def soma(a, b, c):
        idx = cuda.grid(1)
        # TODO: some a[idx] + b[idx] em c[idx] (proteja o limite com if)
        pass

    N = 1_000_000
    a = np.ones(N, dtype=np.float32)
    b = np.ones(N, dtype=np.float32) * 2
    c = np.zeros(N, dtype=np.float32)
    a_d, b_d, c_d = cuda.to_device(a), cuda.to_device(b), cuda.to_device(c)

    tpb = 256
    bpg = (N + tpb - 1) // tpb
    soma[bpg, tpb](a_d, b_d, c_d)          # warm-up
    cuda.synchronize()

    inicio = cuda.event(); fim = cuda.event()
    inicio.record()
    soma[bpg, tpb](a_d, b_d, c_d)
    fim.record(); fim.synchronize()
    print(f"Tempo do kernel: {cuda.event_elapsed_time(inicio, fim):.3f} ms")
    print(f"Correto? {np.allclose(c_d.copy_to_host(), a + b)}")
except Exception as e:
    print(f"Sem GPU CUDA aqui ({e}). Conceito: cada thread calcula c[idx]=a[idx]+b[idx].")


## 8. Síntese e Tarefa de Casa

**O que levar:**
- **Kernel:** função CUDA executada por N threads na GPU.
- **threadIdx/blockIdx:** IDs únicos — usados para calcular o índice global.
- **Grade → Bloco → Thread:** hierarquia de execução; grade = problema inteiro.
- **to_device/copy_to_host:** trânsito RAM ↔ VRAM antes e depois do kernel.
- **synchronize():** aguarda os threads antes de prosseguir na CPU.
- **256 threads/bloco:** ótimo na maioria dos casos (múltiplo de 32).

**Tarefa (opcional):** implemente um kernel CUDA que calcula o **produto escalar** de dois
vetores de 1M elementos e compare com NumPy:
- kernel com `numba.cuda` e versão CPU;
- meça o speedup para N = 1K, 10K, 100K, 1M, 10M;
- plote o gráfico de speedup × N com Matplotlib;
- identifique o ponto em que a GPU supera a CPU.

> 🔗 **Próxima aula:** *Manipulação de Memória em CUDA* — o kernel funciona, mas bate demais
> na VRAM lenta; vamos usar memória compartilhada e tiling.